In [3]:
# Install the necessary solver and modeling framework
!apt-get install -y -qq coinor-cbc
!pip install -q pyomo

import pyomo.environ as pyo

def solve_supply_chain():
    # --- Base Scenario: Minimum Cost Flow ---
    model = pyo.ConcreteModel(name="Capacitated_Network_Flow")

    # Define Nodes and Supply/Demand Data
    nodes = ['BOM', 'BEL', 'SOL', 'HUB', 'MAN', 'GUL', 'BAN', 'CHE']
    supply = {'BOM': 190, 'BEL': 110, 'SOL': 0, 'HUB': 0, 'MAN': 0, 'GUL': 0, 'BAN': -100, 'CHE': -200}

    # Define Edges and Transportation Costs
    edges = {
        ('BOM', 'SOL'): 3, ('BOM', 'HUB'): 2,
        ('BEL', 'HUB'): 4, ('BEL', 'MAN'): 4,
        ('SOL', 'GUL'): 4, ('HUB', 'GUL'): 6,
        ('HUB', 'BAN'): 7, ('MAN', 'BAN'): 3,
        ('GUL', 'CHE'): 2, ('BAN', 'CHE'): 2
    }

    model.N = pyo.Set(initialize=nodes)
    model.E = pyo.Set(initialize=edges.keys())

    # Decision Variables: Flow quantity on each edge
    model.x = pyo.Var(model.E, domain=pyo.NonNegativeReals)

    # Objective: Minimize Total Transportation Cost
    def obj_rule(m):
        return sum(edges[e] * m.x[e] for e in m.E)
    model.obj = pyo.Objective(rule=obj_rule, sense=pyo.minimize)

    # Constraint: Flow Conservation
    def flow_rule(m, n):
        inflow = sum(m.x[i, j] for (i, j) in m.E if j == n)
        outflow = sum(m.x[i, j] for (i, j) in m.E if i == n)
        return inflow - outflow + supply[n] == 0
    model.flow_cons = pyo.Constraint(model.N, rule=flow_rule)

    # Solve Base Scenario
    solver = pyo.SolverFactory('cbc', executable='/usr/bin/cbc') # Explicitly specify the executable path
    solver.solve(model)
    print("--- Base Scenario ---")
    print(f"Optimal Minimum Cost: {pyo.value(model.obj)}")
    for e in model.E:
        if pyo.value(model.x[e]) > 0:
            print(f"{e[0]} -> {e[1]}: {pyo.value(model.x[e])}")

    # --- Scenario 2: Pipeline Disruption & Capacity Limits ---
    model_scen2 = model.clone()

    # Solapur-Gulbarga pipeline is disrupted (capacity = 0)
    model_scen2.x['SOL', 'GUL'].setub(0)

    # Bangalore depot capacity constraint (total inflow <= 150)
    def ban_capacity_rule(m):
        return sum(m.x[i, j] for (i, j) in m.E if j == 'BAN') <= 150
    model_scen2.ban_cap = pyo.Constraint(rule=ban_capacity_rule)

    solver.solve(model_scen2)
    print("\n--- Scenario 2 (Disruption & Capacity Limits) ---")
    print(f"Optimal Minimum Cost: {pyo.value(model_scen2.obj)}")
    for e in model_scen2.E:
        if pyo.value(model_scen2.x[e]) > 0:
            print(f"{e[0]} -> {e[1]}: {pyo.value(model_scen2.x[e])}")

if __name__ == '__main__':
    solve_supply_chain()

ERROR:pyomo.core:Unable to clone Pyomo component attribute.
Component 'E' contains an uncopyable field '_init_values' (<class 'pyomo.core.base.set.TuplizeValuesInitializer'>).  Setting field to `None` on new object


--- Base Scenario ---
Optimal Minimum Cost: 2500.0
BOM -> SOL: 190.0
BEL -> MAN: 110.0
SOL -> GUL: 190.0
MAN -> BAN: 110.0
GUL -> CHE: 190.0
BAN -> CHE: 10.0

--- Scenario 2 (Disruption & Capacity Limits) ---
Optimal Minimum Cost: 2690.0
BOM -> HUB: 190.0
BEL -> MAN: 110.0
HUB -> GUL: 190.0
MAN -> BAN: 110.0
GUL -> CHE: 190.0
BAN -> CHE: 10.0
